In [ ]:
# Match_Parents_To_Soldiers_Partitions_WWI.py
# 11/5/2025

# Match Soldiers Using children of Jewish Cemeteries Parents

# Rank_Dictionary_Lookup.py
# 10/10/2025
from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd


import json

import sys
import random


def maximize_screen():
    from IPython.display import display,HTML
    display(HTML("<style>:root{--jp-notebook-max-width:100%!important;}</style>"))

def  timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':|',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def normalize(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = re.sub(r'[^a-z\s]', '', s)  # keep only letters and spaces
    s = re.sub(r'\s+', ' ', s)      # collapse multiple spaces
    return s.strip()


def haystack_contains_needle(needle_df,haystack_df):
    mask1 = [
    normalize(parent) in normalize(spouse)
    for parent,spouse in zip(
        needle_df,
        haystack_df,  
    )
    ]
    return mask1


def  search_dataframe(dataframe,column,search_text):
    dataframe_extract=dataframe[dataframe[column]==search_text]
    return dataframe_extract

def clean_dataframe(dataframe_df,column):
    delete_list=['Pvt ','PTE ','Dr. ','Rev ','MAJ ','GEN ','Mrs ','CPT ',',PFC ','SGT ','Col ','LT ','Dr ',\
                ' Sr',' Jr','\.','Minor ','Infant ','Boy ','Girl ','Twins ','PFC ','TEC5 ',\
                'PVT ','S Sgt ','Capt ','Pfc ','TSgt ','COL ','Tec5 ','TEC4 ','1','2','III','II','Lt','CPL','SSgt','Sgt',\
                'Staff Sergeant ','TEC 5 ','Maj ','GM3 ','TEC 4 ','Tec4 ','RM3 ','Corp ','T 5 ','MS ','  ']
    dataframe_df=dataframe_df.copy()
    for item in delete_list:
        dataframe_df[column]= dataframe_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return dataframe_df 

def time_lapse(tic):
    toc=datetime.datetime.now()
    processing_time=toc-tic
    processing_time_seconds=int(processing_time.total_seconds())
    processing_time_minutes=processing_time_seconds/60
    return f" {processing_time_minutes:.1f}"

def time_now():
    time_now=datetime.datetime.now()  
    return time_now

def drop_sets_dataframe_duplicates(dataframe_set_df):
    set_columns = [col for col in dataframe_set_df.columns]
    # print(set_columns)
    dataframe_set_df=dataframe_set_df.copy()
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: tuple(sorted(x)) if isinstance(x, set) else x)   
    dataframe_set_df = dataframe_set_df.drop_duplicates()
    dataframe_set_df=dataframe_set_df.copy()
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: set(x) if isinstance(x, tuple) else x )
    return dataframe_set_df
    
def title_case(dataframe_df):
    field_list=list(dataframe_df)
    for column in field_list:
        if not pd.api.types.is_string_dtype(dataframe_df[column]):continue
        dataframe_df[column]=dataframe_df[column].str.title() 
    return dataframe_df
    
def all_parts_in(needle, haystack):
    if pd.isna(haystack) or pd.isna(needle):
        return None
    needle_parts = needle.split()   # split needle into words
    match_parts = all(part in haystack for part in needle_parts)
    return match_parts

def  haystack_contains_needle_fuzzy(dataframe_df,needle,haystack):
    match=haystack+'_match' 
    dataframe_df=dataframe_df.copy()
    dataframe_df[match] = dataframe_df.apply(lambda row: all_parts_in(row[needle], row[haystack]),axis=1)
    mask=dataframe_df[match]
    return dataframe_df, mask

def  haystack_contains_whole_needle_fuzzy(dataframe_df,needle,haystack):
    match=haystack+'_match' 
    dataframe_df=dataframe_df.copy()
    dataframe_df[match] = dataframe_df.apply(lambda row: row[needle] in row[haystack],axis=1)
    mask=dataframe_df[match]
    return dataframe_df, mask

def all_parts_in_haystck(needle, haystack):
    if pd.isna(haystack) or pd.isna(needle):
        return None
    needle_parts = needle   # split needle into words
    match_parts = all(part in haystack for part in needle_parts)
    return match_parts

def expand_rank(full_name):
    for abbreviated_rank, rank in rank_lookup.items():
        pattern = rf"\b{abbreviated_rank}\b"  # match whole code
        if re.search(pattern, full_name.lower()):
            new_full_name=re.sub(pattern, rank, full_name.lower(), count=1)
            return new_full_name.title()
    return full_name  # return unchanged if no code found


def strip_rank(full_name):
    for abbreviated_rank, rank in rank_lookup.items():
        if pd.isna(full_name):return None
        pattern = rf"\b{abbreviated_rank}\b"  # match whole code  
        # print(pattern,'*',full_name)
        if re.search(pattern, full_name):
            new_full_name = re.sub(pattern, "", full_name).strip()  
            # print(new_full_name)
            # input('')
            return  new_full_name
            
    return full_name  # return unchanged if no code found


def extract_rank(full_name):
    for abbreviated_rank, rank in rank_lookup.items():
        pattern = rf"\b{abbreviated_rank}\b"  # match whole code
        if re.search(pattern, full_name.lower()):
            # new_full_name = re.sub(pattern, "", full_name.lower()).strip()           
            return  rank
    return ''  # return unchanged if no code found



def time_lapse(tic):
    toc=datetime.datetime.now()
    processing_time=toc-tic
    processing_time_seconds=int(processing_time.total_seconds())
    return processing_time_seconds

def get_tic():
    tic=datetime.datetime.now()
    return tic

def extract_rank_2(full_name):
    rank_1=''
    rank_2=''
    for abbreviated_rank, rank in rank_lookup.items():
        pattern = rf"\b{abbreviated_rank}\b"  # match whole code
        if re.search(pattern, full_name.lower()):
            full_name = re.sub(pattern, "", full_name.lower()).strip() 
            if rank_1=='':rank_1=rank;continue
            rank_2=rank;return rank_1,rank_2
    return rank_1,rank_2  # return unchanged if no code found

def haystack_contains_needle_dataframe(needle_df0, haystack_df0):
    # Normalize both columns
    global needle_df, haystack_df
    needles = needle_df0.dropna().apply(normalize)
    haystacks = haystack_df0.dropna().apply(normalize)
    # Convert haystack to DataFrame so we can merge vectorized
    hay_df = pd.DataFrame({'haystack_index': haystacks.index, 'haystack_value': haystacks})
    needle_df_ = pd.DataFrame({'needle_index': needles.index, 'needle_value': needles})
    # Use pandas explode trick: build all potential combinations only where likely overlap
    # Here, we use vectorized string contains
    matches = []
    for n_idx, n_val in needle_df_.itertuples(index=False):
        mask = hay_df['haystack_value'].str.contains(n_val, na=False)
        if mask.any():
            subset = hay_df.loc[mask].copy()
            subset['needle_index'] = n_idx
            subset['needle_value'] = n_val
            matches.append(subset)
    if matches:
        result1 = pd.concat(matches, ignore_index=True)
    else:
        result1 = pd.DataFrame(columns=['needle_index', 'needle_value', 'haystack_index', 'haystack_value'])
    merged_df = result1.merge( needle_df, left_on='needle_index', right_index=True).merge(
    haystack_df, left_on='haystack_index', right_index=True,suffixes=('_needle', '_haystack'))
    return merged_df

def Soldier_Parent_Child_Process(soldiers_short_df,parents_short_df):
    print("$$$$$  Soldier-Parent-1-2-Child Processing $$$") 
    global needle_df,haystack_df
    needle_df=soldiers_short_df
    haystack_df=parents_short_df
    parents_soldiers_match_df1 = haystack_contains_needle_dataframe(needle_df['parent_1'], haystack_df['fullName'])
    
    dataframe_df=parents_soldiers_match_df1
    needle='soldiers_full_name'
    haystack='Children'
    parents_soldiers_child_match_df1,mask1=haystack_contains_whole_needle_fuzzy(dataframe_df,needle,haystack)
    parents_soldiers_child_match_df1['rank_match']=(parents_soldiers_child_match_df1['military_rank']==parents_soldiers_child_match_df1['children_rank']) \
                                |(parents_soldiers_child_match_df1['military_rank']==parents_soldiers_child_match_df1['children_rank_2'])
    print('$$$ Parent 1 Processing Complete $$$')
    
    needle_df2=soldiers_short_df[soldiers_short_df['parent_2']!='']
    parents_soldiers_match_df2 = haystack_contains_needle_dataframe(needle_df2['parent_2'], haystack_df['fullName'])
    
    dataframe_df=parents_soldiers_match_df2
    needle='soldiers_full_name'
    haystack='Children'
    parents_soldiers_child_match_df2,mask2=haystack_contains_whole_needle_fuzzy(dataframe_df,needle,haystack)
    parents_soldiers_child_match_df2['rank_match']=parents_soldiers_child_match_df2['military_rank']==parents_soldiers_child_match_df2['children_rank']
    print('$$$ Parent 2 Processing Complete $$$')
    
    parents_soldiers_child_match_df=pd.concat([parents_soldiers_child_match_df1,parents_soldiers_child_match_df2])
    parents_soldiers_child_match_df['Parent']=''
    parents_soldiers_child_match_df['Parent'] = np.where(
        parents_soldiers_child_match_df['fullName'] == parents_soldiers_child_match_df['parent_1'],'parent_1',
        parents_soldiers_child_match_df.get('Parent')  # keep existing value if not matching
    )
    parents_soldiers_child_match_df['Parent'] = np.where(
        parents_soldiers_child_match_df['fullName'] == parents_soldiers_child_match_df['parent_2'],'parent_2',
        parents_soldiers_child_match_df.get('Parent')  # keep existing value if not matching
    )
    
    parents_soldiers_match_df_list =['name_index','soldiers_full_name','Children_match'\
                                   ,'military_rank','rank_match','children_rank','children_rank_2','Children','parent_1','parent_2','Parent','fullName','Spouses','birthState','birthCountry'\
                                   ,'cemeteryName','plot_haystack','cemeteryId','missing_status','birthYear_needle','entry_state','soldiers_cemetery'\
                                   ,'plot_needle','birthCity','lastName','firstName','middleName','intermentHasPhoto'\
                                   ,'birthYear_haystack','birthStateName','birthCountryName','Siblings','soldiers_last_name','soldiers_first_name','soldiers_middle_name']
    
    parents_soldiers_child_match_df=parents_soldiers_child_match_df[parents_soldiers_match_df_list]
    parents_soldiers_child_match_df = parents_soldiers_child_match_df.sort_values(by=['name_index','fullName'],ascending=True)
    print("$$$$$  Soldier-Parent-1-2-Child Processing Complete $$$")
    return parents_soldiers_child_match_df

def get_process_time():
    global tic
    process_time= time_lapse(tic)
    process_time_cum= time_lapse(tic_start)
    tic=get_tic()
    process_time_str=f"{process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes"
    return process_time_str

blank_line='                                                                                                                                                                                           '

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

process_name='Soldier-Parent-Child Processing - WWI' 
now=time_now()
print(f" ****** Starting {process_name} - Time: {now}")
start_time=datetime.datetime.now()
event_time=start_time

# print("***Jewish_Cemetery_Children_Soldiers_Match--- Started *****")
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"

fileABMCInput="ABMC_FAG_Soldiers_Merge_Master"
fileSoDInput="star_david_20251019_df"
fileRankInput="Rank_Dictionary"

fileABMCIn=DirMain+"0-Master Worksheets\\"+fileABMCInput+".csv"
fileSoDIn=DirMain+"0-Master Worksheets\\"+fileSoDInput+".csv"
fileRankIn=DirMain+"0-Master Worksheets\\"+fileRankInput+".csv"
tic_start=get_tic()
tic=get_tic()

soldiers_df=pd.read_csv(fileABMCIn,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
star_david_df=pd.read_csv(fileSoDIn)
rank_df=pd.read_csv(fileRankIn)


rank_lookup = rank_df.set_index("abbreviated_rank")["rank"].to_dict()
soldiers_df = soldiers_df.dropna(subset=['Parents']) 
soldiers_df["soldiers_full_name"]=soldiers_df["soldiers_full_name"].str.lower()
soldiers_df["soldiers_full_name"]=soldiers_df["soldiers_full_name"].astype(str).replace(r' +',' ', regex=True)

soldiers_df=drop_sets_dataframe_duplicates(soldiers_df)
soldiers_df.reset_index(drop=True, inplace = True)

soldiers_df=soldiers_df.copy()

soldiers_df['parent_1']=soldiers_df['Parents'].str.split('*').str[0].str.strip()
soldiers_df['parent_2']=soldiers_df['Parents'].str.split('*').str[1].str.strip()
soldiers_df['parent_1']=soldiers_df['parent_1'].apply(normalize)
soldiers_df['parent_2']=soldiers_df['parent_2'].apply(normalize)


# soldiers_df["soldiers_full_name"]=soldiers_df["soldiers_full_name"].apply(strip_rank)
# process_time= time_lapse(tic)
# process_time_cum= time_lapse(tic_start)
# tic=get_tic()
# print(f"   ---- soldiers_full_name Strip Rank Complete --- - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes",'\n')
# print('Strip:',soldiers_df["soldiers_full_name"])
# input('')


##################################### Partitioned Inputs ###############################################################################################
group_A=['Jewish_Graves','Jewish_Cemeteries_All_Graves_','Reverse_Parents',0,27]
group_WWI=['WWI','FAG_Relationship_Names_WWI_Graves_','WWI-Parents-Match',1,126]

partition_df=pd.DataFrame([group_WWI])
partition_df_list=['directory','format','label','range_start','range_end']
partition_df.columns=[partition_df_list]

parents_df=pd.DataFrame([])
for index, row in partition_df.iterrows():
    subdirectory=row['directory']
    file_format=row['format']
    label=row['label']
    partition_start=row['range_start']
    partition_end=row['range_end']
    fileOutParents=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\"+subdirectory+"\\"+label+"_"+timestamp+".csv"
    start=partition_start
    end=partition_end
    parents_soldiers_match_df=pd.DataFrame([])

    for file_number in range(start,end):
        file_number_str='00'+str(file_number)
        file_number_str=file_number_str[-3:]
        fileInParents=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Inputs\\"+subdirectory+"\\"+file_format+file_number_str+".csv"
        parents_partition_df=pd.read_csv(fileInParents,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
        parents_partition_df_size=len(parents_partition_df)
        parents_partition_df = parents_partition_df.dropna(subset=['Children'])   
        parents_partition_df=drop_sets_dataframe_duplicates(parents_partition_df)
        parents_partition_df.reset_index(drop=True, inplace = True)
        parents_partition_df_file_size=len(parents_partition_df)
        # print('• parents_partition_df * Drop If No Children Entry * Size:',parents_partition_df_file_size) 
        parents_partial_df=parents_partition_df
        parents_partial_df_size=len(parents_partial_df)
        # print('• Normalize parents_partial_df *')
        
        parents_partial_df['Children']=parents_partial_df['Children'].str.lower()
        parents_partial_df['fullName']=parents_partial_df['fullName'].str.lower()     
        parents_partial_df['Children']= parents_partial_df['Children'].astype(str).replace(r'\.', '', regex=True).str.strip()
        parents_partial_df['Children']= parents_partial_df['Children'].astype(str).replace(r' +', ' ', regex=True).str.strip()
        # print("• Extract Children's Rank *")
        
        rank_temp=parents_partial_df["Children"].apply(extract_rank_2) 
        parents_partial_df[['children_rank', 'children_rank_2']] = pd.DataFrame(rank_temp.tolist())
        
        if 'plot' not in parents_partial_df.columns:parents_partial_df['plot']='No Plot Info'
        process_time= time_lapse(tic)
        process_time_cum= time_lapse(tic_start)
        tic=get_tic()      
        print('\n')
        print(f"Group: {label} • Uploading File {file_format+file_number_str} •  Number of Partitions: {end} • Record Count: {parents_partial_df_size:,}/{parents_partition_df_size:,}") 
        
        soldiers_short_df_list=['name_index','soldiers_last_name','soldiers_first_name','soldiers_middle_name','soldiers_full_name'\
                                ,'military_rank','missing_status','birthYear','entry_state'\
                                  ,'soldiers_cemetery','plot','birthCity','birthState','birthCountry','parent_1','parent_2']

        parents_short_df_list=['lastName','firstName','middleName','fullName','cemeteryName','plot','cemeteryId'\
                               ,'intermentHasPhoto','birthYear','birthStateName','birthCountryName','Spouses','Children','children_rank'\
                               ,'children_rank_2','Siblings']
        

       
        parents_short_df=parents_partial_df[parents_short_df_list]
        soldiers_short_df=soldiers_df[soldiers_short_df_list]
        input('')


In [ ]:
print("   ----Soldier-Parent-Child Processing ---") 
tic=get_tic()  
needle_df=soldiers_short_df
haystack_df=parents_short_df
soldiers_parents_child_match_partial_df = haystack_contains_needle_dataframe(needle_df['parent_1'], haystack_df['fullName'])
soldiers_parents_child_match_partial_df_size=len(soldiers_parents_child_match_partial_df)

file_out_df=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\"+subdirectory+"\\"+label+"_soldiers_parents_child_match_partial_df_"+timestamp+".csv"
soldiers_parents_child_match_partial_df.to_csv(file_out_df,mode="w",index=False,header=True)
print(f"   ----Soldier-Parent-Child Processing • Completed • Output Size: {soldiers_parents_child_match_partial_df_size} • Process Time: {get_process_time()}") 


In [14]:
file_out_df=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\"+subdirectory+"\\"+label+"_soldiers_parents_child_match_partial_df_"+timestamp+".csv"
soldiers_parents_child_match_partial_df.to_csv(file_out_df,mode="w",index=False,header=True)
print(f"   ----Soldier-Parent-Child Processing • Completed • Output Size: {soldiers_parents_child_match_partial_df_size} • Process Time: {get_process_time()}") 


   ----Soldier-Parent-Child Processing • Completed • Output Size: 1734 • Process Time: 48.5/62.6 Minutes


In [ ]:
# $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$    


    print('\n')
    print(f"Group: {label} • Uploaded All {end} Partitions • Record Count: {len(parents_df):,}") 
    process_time= time_lapse(tic)
    process_time_cum= time_lapse(tic_start)
    tic=get_tic()
    print(f"$$$$$ Upload Complete $$$ - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes",'\n' ) 
    parents_df["children_rank1"] = parents_df["Children"].apply(lambda x: extract_rank_multiple(x, 1))
    parents_df["children_rank2"] = parents_df["Children"].apply(lambda x: extract_rank_multiple(x, 2))
    process_time= time_lapse(tic)
    process_time_cum= time_lapse(tic_start)
    tic=get_tic()
    print(f"$$$$$ Extracting Children's Rank Complete $$$ - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes",'\n' )    
    # print('Rank1:',parents_df["Children"][parents_df["children_rank1"]!=''],'\n\n')
    # print('Rank2:',parents_df["Children"][parents_df["children_rank2"]!=''])

    print(f" Group Type Uploaded: {label} • Records: {len(parents_df):,} ")        
    file_parents_df=DirMain+"0-Master Worksheets\\Parent-Soldier Processing\\Outputs\\parents_df_"+label+"_"+timestamp+".csv"
    parents_df.to_csv(file_parents_df,mode="w",index=False,header=True)
    print('$$$$$ Partition Downloads Complete $$$\n')

    parents_short_df_list=['lastName','firstName','middleName','fullName','cemeteryName','plot','cemeteryId','intermentHasPhoto','birthYear','birthStateName'\
     ,'birthCountryName','Spouses','Children','children_rank1','children_rank2','Siblings']
    
    soldiers_short_df_list=['name_index','soldiers_last_name','soldiers_first_name','soldiers_middle_name','soldiers_full_name','military_rank','missing_status'\
      ,'birthYear','entry_state','soldiers_cemetery','plot','birthCity','birthState','birthCountry','Parents']
    
    parents_short_df=parents_df[parents_short_df_list]
    soldiers_short_df=soldiers_df[soldiers_short_df_list]
    
    matching_parents_soldiers_df=pd.merge(
        soldiers_short_df,
        parents_short_df,
        left_on=['soldiers_last_name'],
        right_on=['lastName'],    
        how='inner')
    
    matching_parents_soldiers_df_list=['name_index','soldiers_last_name','soldiers_full_name','military_rank','Parents','lastName','fullName'\
                ,'Children','children_rank1','children_rank2','cemeteryName','plot_y','cemeteryId','intermentHasPhoto','Spouses','birthStateName'\
                ,'birthCountryName','missing_status','birthYear_y','soldiers_cemetery'\
                  ,'plot_x','entry_state','birthState','birthCountry','birthYear_x','Siblings']

    process_time= time_lapse(tic)
    process_time_cum= time_lapse(tic_start)
    tic=get_tic()
    print(f"$$$$$ Match parent to Soldier - Complete $$$ - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes",'\n' )   
    
    print(" **** Starting Needle in Haystack: Soldier's Name: ")
    needle='soldiers_full_name'
    haystack='Children'
    
    dataframe_df=matching_parents_soldiers_df[matching_parents_soldiers_df_list]

# $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$    
    dataframe_df=dataframe_df.copy()
    dataframe_df['Children']=dataframe_df['Children'].str.lower()
    dataframe_df['soldiers_full_name']=dataframe_df['soldiers_full_name'].str.lower()
    dataframe_df["soldiers_full_name"]=dataframe_df["soldiers_full_name"].apply(strip_rank)

    matching_parents_soldiers_short_df, mask_name=haystack_contains_needle_fuzzy(dataframe_df,needle,haystack)
    matching_parents_soldiers_short_df=matching_parents_soldiers_short_df[matching_parents_soldiers_short_df['Children_match']==True]
    matching_parents_soldiers_short_df.reset_index(drop=True, inplace = True)

    process_time= time_lapse(tic)
    process_time_cum= time_lapse(tic_start)
    tic=get_tic()
    print(f"$$$$$ Needle in Haystack: Soldier's Name - Complete $$$ - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes",'\n' )       
    
    # First rank Match 
    print(' **** Starting Needle in Haystack: Rank 1 **** ')
    dataframe_df=matching_parents_soldiers_short_df
    needle='military_rank'
    haystack='children_rank1'
    
    matching_parents_soldiers_short_df, mask_rank1=haystack_contains_needle_fuzzy(dataframe_df,needle,haystack)
    
    # Second rank Match 
    print(' **** Starting Needle in Haystack: Rank 2 **** ')
    dataframe_df=matching_parents_soldiers_short_df
    needle='military_rank'
    haystack='children_rank2'
    
    matching_parents_soldiers_short_df, mask_rank2=haystack_contains_needle_fuzzy(dataframe_df,needle,haystack)

    process_time= time_lapse(tic)
    process_time_cum= time_lapse(tic_start)
    tic=get_tic()
    print(f"$$$$$ Needle in Haystack: Childrens Rank 1 and 2 - Complete $$$ - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes",'\n' )       
    
    file_parents_soldier_match=DirMain+"0-Master Worksheets\\Parent-Soldier Processing\\Outputs\\matching_parents_soldiers_df_"+timestamp+".csv"
    matching_parents_soldiers_short_df.to_csv(file_parents_soldier_match,mode="w",index=False,header=True)
    print('$$$$$ matching_parents_soldiers_df Downloads Complete $$$\n')
    
    print(' **** Starting Star_of_David Association ')
    parents_soldiers_match_sod_df=pd.merge(
        matching_parents_soldiers_short_df,
        star_david_df,
        left_on=[ 'name_index'],
        right_on=['sod_Name_Index'],
        how='left')
    
    parents_soldiers_match_sod_df['Star_of_David'] = parents_soldiers_match_sod_df['sod_Name_Index'].fillna('False')
    parents_soldiers_match_sod_df['Star_of_David']=parents_soldiers_match_sod_df['Star_of_David']!='False'
    parents_soldiers_match_sod_df.drop_duplicates(inplace=True)
    
    reorder_list=['name_index','soldiers_last_name','soldiers_full_name','military_rank','Children', 'children_rank1'\
                  ,'children_rank2', 'Children_match','children_rank1_match','children_rank2_match','Star_of_David','Parents'\
                  ,'fullName','Spouses','cemeteryName','plot_y','cemeteryId','birthStateName','birthCountryName','birthYear_y'\
                  ,'missing_status','entry_state','birthState','birthCountry','soldiers_cemetery','birthYear_x','plot_x','Siblings']
    
    rename_list=['name_index','soldiers_last_name','soldiers_full_name','military_rank','Children', 'children_rank1'\
                  ,'children_rank2', 'Children_match','children_rank1_match','children_rank2_match','Star_of_David'\
                 ,'Parents','parent_full_name','spouse','parents_cemetery','plot_parents','cemetery_id'\
                ,'parent_birth_state','parent_birth_country','parent_birth_year'\
                 ,'missing_status','entry_state','birth_state','birth_country','soldiers_cemetery','soldiers_birth_year','soldiers_plot','Siblings']
    
    parents_soldiers_match_sod_df=parents_soldiers_match_sod_df[reorder_list]
    parents_soldiers_match_sod_df.columns=rename_list
    parents_soldiers_match_sod_df['soldiers_full_name']=parents_soldiers_match_sod_df['soldiers_full_name'].str.title()
    parents_soldiers_match_sod_df['Children']=parents_soldiers_match_sod_df['Children'].str.title()
    
    print(' **** Start Saving - parents_soldiers_match_df ')
    file_parents_df=DirMain+"0-Master Worksheets\\Parent-Soldier Processing\\Outputs\\parents_soldiers_match_sod_df_"+label+"_"+timestamp+".csv"
    parents_soldiers_match_sod_df.to_csv(file_parents_df,mode="w",index=False,header=True)
    print('$$$$$ Parent_Soldiers Matching Complete $$$ - Size: ',len(parents_soldiers_match_df))

    process_time= time_lapse(tic)
    process_time_cum= time_lapse(tic_start)
    print(f"$$$$$ Parent-Soldiers SOD - Complete $$$ - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes",'\n' )  


    print(f"$$$$$ Parent-Soldiers Classification  - Complete $$$ - Total Processing Time: {process_time_cum/60:4.1f} Minutes",'\n' )  
    
